In [ ]:
# -*- coding: utf-8 -*-
"""super.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1W03HiuuRYwxK11M-VXiz-B-lblhQdAuc
"""

!pip install -q sentence-transformers faiss-cpu gTTS gradio groq

import os, json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from gtts import gTTS
import gradio as gr
from groq import Groq
from google.colab import drive

drive.mount('/content/drive')

groq_client = Groq(api_key="gsk_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 25.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 1.16.1 which is incompatible.
Mounted at /content/drive


In [ ]:
# 📘 Correct Bhagavad Gita Loader (multilingual-ready)

GITA_VERSE_PATH = "/content/drive/MyDrive/gitadata/verse (1).json"
GITA_TRANS_HI_PATH = "/content/drive/MyDrive/gitadata/translation.json"
GITA_TRANS_EN_PATH = "/content/drive/MyDrive/gitadata/corrected_translation_gpt3.json"

import json

# Load Sanskrit verses
with open(GITA_VERSE_PATH, "r", encoding="utf-8") as f:
    gita_verses_raw = json.load(f)

# Load Hindi translations
with open(GITA_TRANS_HI_PATH, "r", encoding="utf-8") as f:
    gita_hi_raw = json.load(f)

# Load English translations
with open(GITA_TRANS_EN_PATH, "r", encoding="utf-8") as f:
    gita_en_raw = json.load(f)

# Map: verse_id -> Hindi
gita_hi_map = {
    t["verse_id"]: t["description"]
    for t in gita_hi_raw
    if t.get("lang", "").lower() == "hindi"
}

# Map: verse_id -> English
gita_en_map = {
    t["verse_id"]: t["description"]
    for t in gita_en_raw
    if t.get("lang", "").lower() == "english"
}

gita_data = []
for v in gita_verses_raw:
    vid = v["id"]
    gita_data.append({
        "source": "gita",
        "id": str(vid),
        "ref": f"Bhagavad Gita {v['chapter_number']}:{v['verse_number']}",
        "original": v["text"],                          # Sanskrit
        "translation_en": gita_en_map.get(vid, ""),     # for embeddings
        "meta": {
            "chapter": v["chapter_number"],
            "verse_number": v["verse_number"],
            "transliteration": v.get("transliteration", ""),
            "translations": {
                "en": gita_en_map.get(vid, ""),
                "hi": gita_hi_map.get(vid, "")
            }
        }
    })

print("Loaded Bhagavad Gita verses:", len(gita_data))
print("Example:", gita_data[0])

Loaded Bhagavad Gita verses: 701
Example: {'source': 'gita', 'id': '1', 'ref': 'Bhagavad Gita 1:1', 'original': 'धृतराष्ट्र उवाच\n\nधर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः।\n\nमामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय।।1.1।।\n ', 'translation_en': '\nThe King Dhritarashtra asked, "O Sanjaya! What happened on the sacred battlefield of Kurukshetra when my people and the Pandavas gathered?"', 'meta': {'chapter': 1, 'verse_number': 1, 'transliteration': 'dhṛitarāśhtra uvācha\ndharma-kṣhetre kuru-kṣhetre samavetā yuyutsavaḥ\nmāmakāḥ pāṇḍavāśhchaiva kimakurvata sañjaya\n', 'translations': {'en': '\nThe King Dhritarashtra asked, "O Sanjaya! What happened on the sacred battlefield of Kurukshetra when my people and the Pandavas gathered?"', 'hi': '\n\n।।1.1।।धृतराष्ट्र ने कहा -- हे संजय ! धर्मभूमि कुरुक्षेत्र में एकत्र हुए युद्ध के इच्छुक (युयुत्सव:) मेरे और पाण्डु के पुत्रों ने क्या किया?\n\n\n\n\n\xa0'}}}


In [ ]:
QURAN_BASE_DIR = "/content/drive/MyDrive/qurandata"

# --- Load English base (Arabic + English translation) ---
with open(os.path.join(QURAN_BASE_DIR, "quran_en.json"), "r", encoding="utf-8") as f:
    quran_en = json.load(f)

# --- Load other language translations ---
quran_lang_files = {
    "tr": "quran_tr.json",
    "fr": "quran_fr.json",
    "es": "quran_es.json",
    "id": "quran_id.json",
    "ur": "quran_ur.json",
    "ru": "quran_ru.json",
    "sv": "quran_sv.json",
    "zh": "quran_zh.json",
    "bn": "quran_bn.json",
}

quran_translations_by_lang = {}

for lang, fname in quran_lang_files.items():
    path = os.path.join(QURAN_BASE_DIR, fname)
    if not os.path.exists(path):
        print(f"⚠️ Missing Quran file for {lang}: {fname}")
        continue
    with open(path, "r", encoding="utf-8") as f:
        surahs = json.load(f)
    lang_map = {}
    for s in surahs:
        sid = s["id"]
        for v in s["verses"]:
            vid = v["id"]
            lang_map[(sid, vid)] = v.get("translation") or v.get("text")
    quran_translations_by_lang[lang] = lang_map
    print(f"Loaded Quran {lang} translation for {len(lang_map)} verses.")

# --- Load transliteration (note filename spelling: transileration) ---
quran_translit_map = {}
translit_path = os.path.join(QURAN_BASE_DIR, "quran_transliteration.json")
if os.path.exists(translit_path):
    with open(translit_path, "r", encoding="utf-8") as f:
        translit_surahs = json.load(f)
    for s in translit_surahs:
        sid = s["id"]
        for v in s["verses"]:
            vid = v["id"]
            quran_translit_map[(sid, vid)] = v.get("transliteration")
    print("Loaded Quran transliteration entries:", len(quran_translit_map))
else:
    print("⚠️ No quran_transileration.json found.")

# --- Build unified Quran verse list ---
quran_data = []

for s in quran_en:
    sid = s["id"]
    surah_name_ar = s["name"]
    surah_name_en = s["translation"]
    surah_name_translit = s["transliteration"]

    for v in s["verses"]:
        vid = v["id"]
        arabic_text = v["text"]
        eng_translation = v["translation"]

        # Collect all available translations for this verse
        lang_translations = {"en": eng_translation}
        for lang, mapping in quran_translations_by_lang.items():
            t = mapping.get((sid, vid))
            if t:
                lang_translations[lang] = t

        transliteration = quran_translit_map.get((sid, vid))

        quran_data.append({
            "source": "quran",
            "id": f"{sid}:{vid}",
            "ref": f"Quran {surah_name_translit} ({sid}:{vid})",
            "original": arabic_text,
            "translation_en": eng_translation,
            "meta": {
                "surah_id": sid,
                "verse_id": vid,
                "surah_name_ar": surah_name_ar,
                "surah_name_en": surah_name_en,
                "surah_name_translit": surah_name_translit,
                "translations": lang_translations,        # dict lang -> text
                "transliteration": transliteration        # may be None
            }
        })

print("Loaded Quran verses:", len(quran_data))

Loaded Quran tr translation for 6236 verses.
Loaded Quran fr translation for 6236 verses.
Loaded Quran es translation for 6236 verses.
Loaded Quran id translation for 6236 verses.
Loaded Quran ur translation for 6236 verses.
Loaded Quran ru translation for 6236 verses.
Loaded Quran sv translation for 6236 verses.
Loaded Quran zh translation for 6236 verses.
Loaded Quran bn translation for 6236 verses.
Loaded Quran transliteration entries: 6236
Loaded Quran verses: 6236


In [ ]:
BIBLE_PATH = "/content/drive/MyDrive/bibledata/verses-1769.json"

with open(BIBLE_PATH, "r", encoding="utf-8") as f:
    bible_raw = json.load(f)

bible_data = []
for ref, text in bible_raw.items():
    bible_data.append({
        "source": "bible",
        "id": ref,
        "ref": ref,
        "original": text,          # base text
        "translation_en": text,    # for embeddings
        "meta": {
            "translations": {
                "en": text        # only English; others will be AI-translated
            }
        }
    })

print("Loaded Bible verses:", len(bible_data))
print("Example:", bible_data[0])

all_verses = gita_data + quran_data + bible_data
print("Total verses from all scriptures:", len(all_verses))

# SentenceTransformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

texts_for_embeddings = [v["translation_en"] for v in all_verses]
embeddings = model.encode(texts_for_embeddings, convert_to_numpy=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index ready. Dimension:", dimension)

model = SentenceTransformer("all-MiniLM-L6-v2")

texts_for_embeddings = [v["translation_en"] for v in all_verses]

embeddings = model.encode(texts_for_embeddings, convert_to_numpy=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index ready.")

Loaded Bible verses: 31102
Example: {'source': 'bible', 'id': 'Genesis 1:1', 'ref': 'Genesis 1:1', 'original': 'In the beginning God created the heaven and the earth.', 'translation_en': 'In the beginning God created the heaven and the earth.', 'meta': {'translations': {'en': 'In the beginning God created the heaven and the earth.'}}}
Total verses from all scriptures: 38039


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index ready. Dimension: 384


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index ready.


In [ ]:
def search_verses(query, top_k=3, scripture_choice="All"):
    q_emb = model.encode([query], convert_to_numpy=True)
    D, I = index.search(q_emb, top_k * 7)

    wanted = None
    if scripture_choice == "Bhagavad Gita": wanted = {"gita"}
    if scripture_choice == "Quran": wanted = {"quran"}
    if scripture_choice == "Bible": wanted = {"bible"}

    results = []
    for idx in I[0]:
        verse = all_verses[idx]
        if wanted and verse["source"] not in wanted:
            continue
        if verse not in results:
            results.append(verse)
        if len(results) == top_k:
            break

    return results

def generate_master_english_explanation(user_query, verses):
    verse_block = ""
    for v in verses:
        verse_block += f"{v['ref']} — {v['translation_en']}\n"

    prompt = f"""
You are a spiritual guide.

USER QUESTION:
{user_query}

SCRIPTURE VERSES:
{verse_block}

TASK:
Write the BEST POSSIBLE explanation in ENGLISH only.
Do NOT translate.
Do NOT include other languages.
"""

    res = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role":"user","content":prompt}],
        max_tokens=2000, temperature=0.7
    )
    return res.choices[0].message.content

def translate_text(text, target_lang):
    if target_lang == "en":
        return text

    prompt = f"""
Translate the following English spiritual explanation into {target_lang}.
Keep the meaning exactly same. No extra sentences.

TEXT:
{text}
"""

    res = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role":"user","content":prompt}],
        max_tokens=2000, temperature=0.4
    )
    return res.choices[0].message.content

def build_display_text(user_query, verses, answer_lang="en"):
    reply = "🌸 **Spiritual Guidance** 🌸\n\n"
    reply += f"**Your question:** {user_query}\n\n"
    reply += "📖 **Relevant verses:**\n\n"

    for v in verses:
        src = v["source"]
        meta = v.get("meta", {})

        if src == "quran":
            m = meta
            shown_translation = pick_translation(
                m,
                v["translation_en"],
                answer_lang
            )
            reply += (
                f"🕌 *Quran – {m['surah_name_translit']} ({m['surah_id']}:{m['verse_id']})*\n"
                f"Arabic: {v['original']}\n"
                f"Translation: {shown_translation}\n\n"
            )

        elif src == "gita":
            shown_translation = pick_translation(
                meta,
                v["translation_en"],
                answer_lang
            )
            reply += (
                f"🕉 *Bhagavad Gita – {v['ref']}*\n"
                f"Sanskrit: {v['original']}\n"
                f"Transliteration: {meta.get('transliteration', '')}\n"
                f"Translation: {shown_translation}\n\n"
            )

        elif src == "bible":
            # currently only English in meta['translations']
            shown_translation = pick_translation(
                meta,
                v["translation_en"],
                answer_lang
            )
            reply += (
                f"✝ *Bible – {v['ref']}*\n"
                f"Text: {shown_translation}\n\n"
            )

    return reply

In [ ]:
# REPLACE THIS ENTIRE CELL CONTENT
def scripture_response(user_query, scripture_choice, answer_lang):
    # The helper function pick_translation is assumed to be defined elsewhere if needed.
    # Otherwise, you may get a NameError.

    verses = search_verses(user_query, 3, scripture_choice)

    if not verses:
        # Crucial change: Return three empty values if no verses are found
        return "No verses found.", None, None

    # 🔥 Build a full verse display with originals + translations
    verse_display = "## 📖 Relevant Verses From Scriptures\n\n"

    for v in verses:
        src = v["source"]
        meta = v.get("meta", {})

        # --- Verse Text Block ---
        if src == "gita":
            verse_display += (
                f"### 🕉 Bhagavad Gita — {v['ref']}\n"
                f"**Sanskrit:**\n{v['original']}\n\n"
                f"**Transliteration:**\n{meta.get('transliteration','')}\n\n"
                f"**Translation ({answer_lang}):**\n{meta['translations'].get(answer_lang, meta['translations']['en'])}\n\n"
            )

        elif src == "quran":
            m = meta
            verse_display += (
                f"### 🕌 Quran — {m['surah_name_translit']} ({m['surah_id']}:{m['verse_id']})\n"
                f"**Arabic:**\n{v['original']}\n\n"
                f"**Translation ({answer_lang}):**\n{m['translations'].get(answer_lang, m['translations']['en'])}\n\n"
            )

        elif src == "bible":
            verse_display += (
                f"### ✝ Bible — {v['ref']}\n"
                f"**Original Text:**\n{v['original']}\n\n"
                f"**Translation ({answer_lang}):**\n{v['translation_en']}\n\n"
            )
        # --- End Verse Text Block ---

    # 🔥 English master explanation
    english_exp = generate_master_english_explanation(user_query, verses)

    # 🔥 Translate explanation
    translated_exp = translate_text(english_exp, answer_lang)

    # 🔥 1. Final text for display in the main Textbox
    display_output = (
        verse_display +
        "\n---\n" +
        "## 🌼 English Explanation\n" +
        english_exp +
        "\n\n## 🌍 Translated Explanation (" + answer_lang + ")\n" +
        translated_exp
    )

    # 🔥 2. Text for the recitation audio (Use the original/transliteration of the first verse)
    verse_text_for_audio = ""
    if verses:
        first_verse = verses[0]
        if first_verse['source'] == 'gita' and first_verse['meta'].get('transliteration'):
            verse_text_for_audio = first_verse['meta']['transliteration']
        else:
            verse_text_for_audio = first_verse['original']

    # 🔥 3. Text for the explanation audio (The final translated explanation)
    explanation_text_for_audio = translated_exp

    # Return all three parts
    return display_output, verse_text_for_audio, explanation_text_for_audio

In [ ]:
# REPLACE THE ENTIRE FINAL GRADIO CELL WITH THIS CODE

LANGS = {
    "English": "en", "Hindi": "hi", "Telugu": "te", "Tamil": "ta",
    "Urdu":"ur","Kannada":"kn","Malayalam":"ml","Gujarati":"gu",
    "Marathi":"mr","Bengali":"bn","Arabic":"ar","French":"fr","Spanish":"es"
}


def ui_wrapper(msg, scr, lang_label):
    code = LANGS[lang_label]

    try:
        display_text, verse_text, explanation_text = scripture_response(msg, scr, code)
    except Exception as e:
        return f"An error occurred: {e}", None, None

    # --- 2. Generate Verse Audio (Recitation) ---
    verse_audio_path = None
    if verse_text:
        try:
            verse_lang_code = 'en'
            if 'quran' in scr.lower():
                verse_lang_code = 'ar'
            elif 'gita' in scr.lower():
                verse_lang_code = 'hi'

            verse_tts = gTTS(text=verse_text, lang=verse_lang_code)
            verse_audio_path = "/tmp/verse_out.mp3"
            verse_tts.save(verse_audio_path)
        except Exception:
            pass

    # --- 3. Generate Explanation Audio ---
    explanation_audio_path = None
    if explanation_text:
        try:
            explanation_tts = gTTS(text=explanation_text, lang=code)
            explanation_audio_path = "/tmp/explanation_out.mp3"
            explanation_tts.save(explanation_audio_path)
        except Exception:
            pass

    return display_text, verse_audio_path, explanation_audio_path

# ----------------- HIGHLY THEMED GRADIO BLOCKS -----------------

custom_css = """
    /* Main Background: Deep, calming indigo */
    .gradio-container {
        background: #1C1F33;
        color: #E0E0E0;
        max-width: 1100px;
        margin: auto;
        border-radius: 20px;
        box-shadow: 0 15px 30px rgba(0, 0, 0, 0.4);
    }

    /* Headers and Title: Soothing Gold/Cream */
    h1, h2, h3 {
        color: #FFD700 !important; /* Gold */
        padding-top: 10px; /* Ensure space above headings for clarity */
    }

    /* Input/Output Boxes: Darker background, soft edges */
    .gr-box, .gr-textbox, .gr-dropdown, .gr-button {
        border-radius: 12px !important;
        border: 1px solid #4A4E69 !important;
        background-color: #2A2E40 !important;
        color: #E0E0E0 !important;
    }

    /* Primary Button: Gold Accent */
    .btn {
        background-color: #FFD700 !important; /* Gold */
        color: #1C1F33 !important; /* Deep Indigo text */
        font-weight: bold;
        transition: background-color 0.2s ease;
    }

    /* Markdown Text Styling (Ensure proper spacing and visibility for explanations) */
    .markdown-output {
        padding: 15px;
        background-color: #2A2E40;
        border-radius: 12px;
        border: 1px solid #4A4E69;
        line-height: 1.6;
    }
    .markdown-output hr {
        border-top: 1px solid #FFD700; /* Use gold separator line */
    }

    /* Center the main title */
    .markdown-output h1 {
        text-align: center !important;
    }
"""

with gr.Blocks(
    theme=gr.themes.Monochrome(),
    css=custom_css
) as demo:

    gr.Markdown("# 🙏 Multi-Scripture Spiritual Guide 🙏")

    ask_button = None

    with gr.Row(variant="panel"):

        # --- INPUT COLUMN ---
        with gr.Column(scale=1, min_width=300):
            gr.Markdown("## ✍️ Ask Your Question")
            msg = gr.Textbox(label="Topic", lines=3, placeholder="E.g., How can I control my anger?", elem_id="question_input")

            with gr.Row():
                scr = gr.Dropdown(["All", "Bhagavad Gita", "Quran", "Bible"], value="All", label="Scripture Source")
                lang = gr.Dropdown(list(LANGS.keys()), value="English", label="Answer Language")

            ask_button = gr.Button("Seek Eternal Wisdom 🌟", elem_classes="btn")

        # --- OUTPUT COLUMN ---
        with gr.Column(scale=2):
            gr.Markdown("## 📜 Guidance from the Holy Texts")
            output_text = gr.Markdown(label="Guidance and Explanation", elem_classes="markdown-output")

    # Audio Outputs Row below the main response
    gr.Markdown("## 🎧 Hear the Recitation and Explanation")
    with gr.Row():
        output_verse_audio = gr.Audio(label="Verse Recitation (Original Language)", type="filepath")
        output_explanation_audio = gr.Audio(label="Explanation Audio (Translated)", type="filepath")

    # --- Launch Logic ---
    ask_button.click(
        ui_wrapper,
        inputs=[msg, scr, lang],
        outputs=[output_text, output_verse_audio, output_explanation_audio]
    )

    demo.load(
        fn=lambda: "Welcome! Ask any question about life, ethics, or spirituality. The wisdom of the world's great scriptures awaits.",
        outputs=[output_text]
    )

demo.launch(share=True)

/tmp/ipykernel_866/2210944154.py:99: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a77131afbd5c217f94.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
# --- CELL 16: YOUR CUSTOM EMOTION TRAINER ---
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
import joblib

# 1. Load the dataset
df = pd.read_csv('/content/drive/MyDrive/majorprojectdata/emotiondata/go_emotions.csv')

# 2. Define the emotions we care about
emotions = ['anger', 'annoyance', 'caring', 'confusion', 'disappointment', 'fear', 'grief', 'joy', 'love', 'nervousness', 'sadness', 'optimism']

def get_label(row):
    for e in emotions:
        if row[e] == 1: return e
    return 'neutral'

df['target'] = df.apply(get_label, axis=1)
df_clean = df[df['target'] != 'neutral'] # Remove neutral to make it more sensitive

# 3. Train with N-grams (makes it understand context like "not happy")
vectorizer = TfidfVectorizer(max_features=2000, stop_words='english', ngram_range=(1,2))
X = vectorizer.fit_transform(df_clean['text'])
y = df_clean['target']

# Train the model
emotion_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42)
emotion_clf.fit(X, y)

# 4. Save
joblib.dump(emotion_clf, '/content/drive/MyDrive/my_emotion_model.pkl')
joblib.dump(vectorizer, '/content/drive/MyDrive/my_vectorizer.pkl')

print("✅ Model Trained! Ready for submission.")

✅ Model Trained! Ready for submission.


In [10]:
# --- CELL 16.5: HYBRID EMOTION VALIDATOR (THE "SAFETY NET") ---
from transformers import pipeline
import numpy as np

# 1. Load a high-accuracy pretrained model (DistilBERT is fast and small)
# This model is trained on 6 basic emotions: sadness, joy, love, anger, fear, surprise
try:
    safety_net_model = pipeline("text-classification", model="bhadresh-savani/distilbert-base-uncased-emotion", top_k=1)
    print("✅ Pretrained Safety Net Loaded.")
except:
    print("⚠️ Transformer loading failed. Ensure 'transformers' and 'torch' are installed.")

def get_hybrid_emotion(text, custom_clf, custom_vec):
    """
    Predicts emotion using the custom model first,
    then validates/corrects with the pretrained model.
    """
    # Step A: Your Custom Model Prediction
    vec = custom_vec.transform([text])
    custom_probs = custom_clf.predict_proba(vec)[0]
    max_prob = np.max(custom_probs)
    custom_emotion = custom_clf.classes_[np.argmax(custom_probs)]

    # Step B: Check Confidence
    # If confidence is low (e.g., < 40%), we ask the 'Expert' model
    if max_prob < 0.40:
        result = safety_net_model(text)[0][0]
        # Map DistilBERT names to your labels if they differ
        expert_emotion = result['label']
        return f"{expert_emotion} (refined)"

    return custom_emotion

print("Hybrid Engine Ready. You can now hide this cell during the demo!")

config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Pretrained Safety Net Loaded.
Hybrid Engine Ready. You can now hide this cell during the demo!


In [11]:
# --- CELL 17: EMOTION + SCRIPTURE MUSIC LOGIC ---
import joblib

# Load your custom-trained models
my_clf = joblib.load('/content/drive/MyDrive/my_emotion_model.pkl')
my_vec = joblib.load('/content/drive/MyDrive/my_vectorizer.pkl')

# MUSIC DATABASE (Using direct web links for safety)
# Sad/Angry -> Calm Music | Joy/Love -> Happy Music
MUSIC_LINKS = {
    "Gita_Sad": "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-8.mp3", # Spiritual Flute type
    "Gita_Happy": "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3", # Joyful Sitar type
    "Universal_Sad": "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-3.mp3", # Soothing Piano
    "Universal_Happy": "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-2.mp3" # Uplifting
}

def ui_wrapper_final(msg, scr, lang_label):
    # 1. Detect Emotion with YOUR model
    vec = my_vec.transform([msg])

    emotion = get_hybrid_emotion(msg, my_clf, my_vec)

    # 2. Pick Music based on BOTH Emotion and Scripture choice
    is_sad = emotion in ['sadness', 'grief', 'fear', 'anger', 'nervousness']

    if "Gita" in scr:
        bg_music = MUSIC_LINKS["Gita_Sad"] if is_sad else MUSIC_LINKS["Gita_Happy"]
    else:
        bg_music = MUSIC_LINKS["Universal_Sad"] if is_sad else MUSIC_LINKS["Universal_Happy"]

    # 3. Get Scripture Response
    display_text, verse_text, explanation_text = scripture_response(msg, scr, LANGS[lang_label])

    # Header for the UI
    status = f"### 🧠 AI Analysis: **{emotion.upper()}** | 🎵 Playing: **Healing Audio**\n\n---\n"
    final_display = status + display_text

    # 4. Generate TTS
    v_path, e_path = "/content/v.mp3", "/content/e.mp3"
    v_lang = 'hi' if "Gita" in display_text else 'ar' if "Quran" in display_text else 'en'
    gTTS(text=verse_text, lang=v_lang).save(v_path)
    gTTS(text=explanation_text, lang=LANGS[lang_label]).save(e_path)

    return final_display, v_path, e_path, bg_music

In [ ]:
# --- CELL 18: FINAL GRADIO INTERFACE ---
import gradio as gr

with gr.Blocks(css=".header {text-align:center; color:#2E7D32} .btn {background:#2E7D32 !important; color:white !important}") as demo:
    gr.HTML("<div class='header'><h1>🕊️ Spiritual Healer & Emotion AI</h1><p>Major Project Submission v5.0</p></div>")

    with gr.Row():
        with gr.Column():
            msg = gr.Textbox(label="Express your feelings...", lines=3)
            scr = gr.Dropdown(["All", "Bhagavad Gita", "Quran", "Bible"], value="All", label="Wisdom Source")
            lang = gr.Dropdown(list(LANGS.keys()), value="English", label="Output Language")
            btn = gr.Button("Seek Wisdom & Music", elem_classes="btn")

        with gr.Column():
            out_txt = gr.Markdown()

    with gr.Row():
        out_v = gr.Audio(label="Verse Recitation", type="filepath")
        out_e = gr.Audio(label="Explanation Audio", type="filepath")
        out_m = gr.Audio(label="Contextual Healing Music", type="filepath", autoplay=True)

    btn.click(ui_wrapper_final, [msg, scr, lang], [out_txt, out_v, out_e, out_m])

demo.launch(debug=True, share=True)

/tmp/ipykernel_866/3899049309.py:4: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=".header {text-align:center; color:#2E7D32} .btn {background:#2E7D32 !important; color:white !important}") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fc75053b49f0922074.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
